# generative_model.py converted
GraphVAE model definition and tests converted for interactive exploration.
The large model code is provided in a single code cell.

In [1]:
"""
Graph Variational Autoencoder (GraphVAE) for EGFR ligand generation.

Architecture:
- Encoder: Compresses PyG graphs into latent vectors (mu, logvar)
- Decoder: Reconstructs node features and edge matrices from latent + 3D pocket info
- Sampling: Reparameterization trick for differentiable sampling
- Conditioning Hook: T790M pocket geometry injection for structure-based generation

The generative model will eventually extend the discriminative GNN regressor
to enable structure-based generation conditioned on binding pocket geometry.
"""

from __future__ import annotations

from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool, GraphConv
from torch_geometric.data import Data, Batch


class GraphEncoder(nn.Module):
    """
    Encodes a batch of PyG graphs into latent vectors.
    """
    
    def __init__(
        self,
        node_features: int = 6,
        edge_features: int = 4,
        global_features: int = 2,
        hidden_dim: int = 128,
        latent_dim: int = 64,
        num_layers: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.node_encoder = nn.Linear(node_features, hidden_dim)
        self.edge_encoder = nn.Linear(edge_features, hidden_dim)
        
        # Graph convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp))
        
        self.dropout = nn.Dropout(dropout)
        self.latent_dim = latent_dim
        
        # Project graph embeddings to latent space (mu and logvar)
        self.to_mu = nn.Linear(hidden_dim + global_features, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim + global_features, latent_dim)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        batch: torch.Tensor,
        global_features: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (mu, logvar) for reparameterization sampling."""
        # Encode node and edge features
        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)
        
        # Apply graph convolutions
        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
            x = F.relu(x)
            x = self.dropout(x)
        
        # Aggregate to graph level
        graph_emb = global_mean_pool(x, batch)
        
        # Handle global features shape: could be concatenated [g1_f1, g1_f2, g2_f1, g2_f2, ...]
        # Reshape to (num_graphs, 2) if needed
        num_graphs = graph_emb.size(0)
        if global_features.size(0) != num_graphs:
            # Flatten and reshape to (num_graphs, features_per_graph)
            global_features = global_features.view(num_graphs, -1)
        elif global_features.dim() == 1:
            global_features = global_features.unsqueeze(0)
        
        # Concatenate with global features
        graph_emb = torch.cat([graph_emb, global_features], dim=-1)
        
        # Project to latent space
        mu = self.to_mu(graph_emb)
        logvar = self.to_logvar(graph_emb)
        
        return mu, logvar


class GraphDecoder(nn.Module):
    """
    Decodes latent vectors back into graph structure and node attributes.
    """
    
    def __init__(
        self,
        latent_dim: int = 64,
        hidden_dim: int = 128,
        pocket_dim: int = 128,  # Actual pocket embedding dimension from pocket_extraction.py
        max_nodes: int = 50,
        node_features: int = 6,
        edge_features: int = 4,
        dropout: float = 0.2,
        use_pocket_conditioning: bool = True,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.pocket_dim = pocket_dim
        self.max_nodes = max_nodes
        self.hidden_dim = hidden_dim
        self.use_pocket_conditioning = use_pocket_conditioning
        
        # Handles both conditioned (z + pocket_embedding) and unconditioned (z only)
        # When use_pocket_conditioning=True and pocket_embedding is provided in forward,
        # the input will be concatenated to (latent_dim + pocket_dim)
        input_dim = latent_dim
        
        self.mlp_expand = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        # Optional projection layer for pocket embedding when conditioning is used
        if use_pocket_conditioning:
            self.pocket_projector = nn.Linear(pocket_dim, hidden_dim // 2)
            self.mlp_expand[0] = nn.Linear(input_dim + hidden_dim // 2, hidden_dim)
        
        # Decode node features: (hidden_dim) -> (max_nodes, node_features)
        self.node_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, max_nodes * node_features),
        )
        
        # Decode edge adjacency: (hidden_dim) -> (max_nodes * max_nodes)
        self.edge_adjacency_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, max_nodes * max_nodes),
        )
        
        # Decode edge types (bond types): (hidden_dim) -> (max_nodes * max_nodes * edge_features)
        self.edge_type_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, max_nodes * max_nodes * edge_features),
        )
    
    def forward(
        self,
        z: torch.Tensor,
        pocket_embedding: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Decode latent vectors to graph structure.
        """
        batch_size = z.size(0)
        
        # === 3D POCKET CONDITIONING INJECTION ===
        # When pocket_embedding is provided, project it and concatenate with latent z
        if self.use_pocket_conditioning and pocket_embedding is not None:
            # Project pocket embedding to intermediate dimension
            pocket_proj = self.pocket_projector(pocket_embedding)  # (batch, hidden_dim//2)
            # Concatenate with latent code
            h_input = torch.cat([z, pocket_proj], dim=-1)  # (batch, latent_dim + hidden_dim//2)
            h = self.mlp_expand(h_input)
        else:
            # Unconditional generation: use latent z directly
            h = self.mlp_expand(z)
        
        # Decode to node features
        node_logits_flat = self.node_decoder(h)
        node_logits = node_logits_flat.view(batch_size, self.max_nodes, 6)
        
        # Decode to edge adjacency
        edge_adj_flat = self.edge_adjacency_decoder(h)
        edge_adjacency = edge_adj_flat.view(batch_size, self.max_nodes, self.max_nodes)
        
        # Decode to edge types
        edge_type_flat = self.edge_type_decoder(h)
        edge_type_logits = edge_type_flat.view(batch_size, self.max_nodes, self.max_nodes, 4)
        
        return node_logits, edge_adjacency, edge_type_logits


class GraphVAE(nn.Module):
    """
    Full Graph Variational Autoencoder for EGFR ligand generation.
    """
    
    def __init__(
        self,
        node_features: int = 6,
        edge_features: int = 4,
        global_features: int = 2,
        hidden_dim: int = 128,
        latent_dim: int = 64,
        num_encoder_layers: int = 3,
        max_nodes: int = 50,
        pocket_dim: int = 128,
        dropout: float = 0.2,
        beta: float = 1.0,  # KL divergence weight in ELBO
        use_pocket_conditioning: bool = True,  # Enable 3D pocket conditioning
    ):
        super().__init__()
        
        self.encoder = GraphEncoder(
            node_features=node_features,
            edge_features=edge_features,
            global_features=global_features,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            num_layers=num_encoder_layers,
            dropout=dropout,
        )
        
        self.decoder = GraphDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            pocket_dim=pocket_dim,
            max_nodes=max_nodes,
            node_features=node_features,
            edge_features=edge_features,
            dropout=dropout,
            use_pocket_conditioning=use_pocket_conditioning,
        )
        
        self.latent_dim = latent_dim
        self.beta = beta  # Weight on KL term in ELBO
    
    def encode(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        batch: torch.Tensor,
        global_features: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode graph batch to (mu, logvar)."""
        return self.encoder(x, edge_index, edge_attr, batch, global_features)
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample from N(mu, exp(logvar)) using reparameterization trick."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def decode(
        self,
        z: torch.Tensor,
        pocket_embedding: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Decode latent vectors to graph structure."""
        return self.decoder(z, pocket_embedding=pocket_embedding)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        batch: torch.Tensor,
        global_features: torch.Tensor,
        pocket_embedding: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Full VAE forward pass.
        """
        # Encode
        mu, logvar = self.encode(x, edge_index, edge_attr, batch, global_features)
        
        # Reparameterize
        z = self.reparameterize(mu, logvar)
        
        # Decode
        node_logits, edge_adjacency, edge_type_logits = self.decode(z, pocket_embedding)
        
        return node_logits, edge_adjacency, edge_type_logits, mu, logvar
    
    def generate(
        self,
        num_samples: int,
        device: str = 'cpu',
        pocket_embedding: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Generate new graphs by sampling from the prior N(0, I).
        """
        z = torch.randn(num_samples, self.latent_dim, device=device)
        return self.decode(z, pocket_embedding)


# ============================================================================
# PLACEHOLDER: POCKET ENCODER (for future integration with 3D geometry)
# ============================================================================

if __name__ == "__main__":
    ...

/Users/ameliaburton/Downloads/clean data/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
